In [ ]:
!pip install docling easyocr azure-ai-documentintelligence azure-core pymupdf matplotlib python-dotenv openai

In [6]:
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from dotenv import load_dotenv
import os
load_dotenv()

endpoint = os.getenv("FORM_RECOGNIZER_ENDPOINT")
key = os.getenv("FORM_RECOGNIZER_KEY")


In [7]:
# formUrl = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/invoice_sample.jpg"
# formUrl = "https://s3-jetl.s3.us-east-2.amazonaws.com/Invoice+%23INV-2026-00847.pdf"
formUrl = "https://s3-jetl.s3.us-east-2.amazonaws.com/Invoice-2.pdf"


In [8]:
document_intelligence_client  = DocumentIntelligenceClient(
    endpoint=endpoint, credential=AzureKeyCredential(key)
)

poller = document_intelligence_client.begin_analyze_document(
    "prebuilt-invoice", AnalyzeDocumentRequest(url_source=formUrl)
)
invoices = poller.result()

In [ ]:
# from docling.document_converter import DocumentConverter, PdfFormatOption, InputFormat
# from docling.datamodel.pipeline_options import PdfPipelineOptions, EasyOcrOptions
# from docling_core.types.doc import ImageRefMode
# from pathlib import Path

# output_dir = Path("./output")
# output_dir.mkdir(parents=True, exist_ok=True)

# pipeline_options = PdfPipelineOptions()
# pipeline_options.do_table_structure = True
# pipeline_options.do_ocr = True
# pipeline_options.ocr_options = EasyOcrOptions(
#     force_full_page_ocr=False,
#     lang=["en"]
# )
# pipeline_options.generate_picture_images = True

# converter = DocumentConverter(
#     format_options={
#         InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
#     }
# )

# result = converter.convert(formUrl)
# doc = result.document

# # Save extracted images to disk with predictable filenames
# for pic in doc.pictures:
#     if pic.image and pic.image.pil_image:
#         img_filename = output_dir / f"{pic.self_ref.replace('/', '_')}.png"
#         pic.image.pil_image.save(img_filename)
#         # Point the reference to the saved file path
#         pic.image.uri = img_filename

# # Export to markdown with file references instead of base64
# markdown_text = doc.export_to_markdown(image_mode=ImageRefMode.REFERENCED)

# # Save markdown to disk
# output_md = output_dir / "output.md"
# output_md.write_text(markdown_text, encoding="utf-8")

# print(markdown_text[:10])

In [4]:
import json

def get_bounding_info(field):
    """Returns list of {page, bbox} dicts."""
    regions = getattr(field, "bounding_regions", None) or []
    result = []
    for r in regions:
        p = r.polygon
        coords = [(p[i], p[i+1]) for i in range(0, len(p), 2)]
        result.append({"page": r.page_number, "bbox": coords})
    return result


def field_to_dict(f, getter):
    return {
        "value": getter(f),
        "confidence": f.confidence,
        "location": get_bounding_info(f),
    }

def string_to_dict(f):
    return field_to_dict(f, lambda x: x.value_string)

def date_to_dict(f):
    return field_to_dict(f, lambda x: str(x.value_date))

def currency_to_dict(f):
    return {
        "value": f.value_currency.amount,
        "currency": f.value_currency.currency_code,
        "confidence": f.confidence,
        "location": get_bounding_info(f),
    }

def address_to_dict(f):
    addr = f.value_address
    structured = vars(addr) if hasattr(addr, "__dict__") else {}
    return {
        "value": structured,
        "raw": getattr(f, "content", None),
        "confidence": f.confidence,
        "location": get_bounding_info(f),
    }


# ── Pick currency code from first available currency field ──────────
def get_currency_code(invoice):
    for field_name in ["InvoiceTotal", "SubTotal", "TotalTax", "AmountDue", "PreviousUnpaidBalance"]:
        f = invoice.fields.get(field_name)
        if f and f.value_currency and f.value_currency.currency_code:
            return f.value_currency.currency_code
    return None


invoice = invoices.documents[0]
inv = {}

# ── Currency (top-level, derived from first available currency field) ──
inv["Currency"] = get_currency_code(invoice)

# ── String fields ──────────────────────────────────────────────
for field_name in [
    "VendorName", "VendorAddressRecipient",
    "CustomerName", "CustomerId", "CustomerAddressRecipient",
    "BillingAddressRecipient", "ShippingAddressRecipient",
    "RemittanceAddressRecipient", "ServiceAddressRecipient",
    "PurchaseOrder", "InvoiceId",
    "VendorTaxId", "CustomerTaxId",
    "PaymentTerm", "KVKNumber",
]:
    f = invoice.fields.get(field_name)
    if f:
        inv[field_name] = string_to_dict(f)

# ── Address fields ─────────────────────────────────────────────
for field_name in [
    "VendorAddress", "CustomerAddress",
    "BillingAddress", "ShippingAddress",
    "RemittanceAddress", "ServiceAddress",
]:
    f = invoice.fields.get(field_name)
    if f:
        inv[field_name] = address_to_dict(f)

# ── Date fields ────────────────────────────────────────────────
for field_name in ["InvoiceDate", "DueDate", "ServiceStartDate", "ServiceEndDate"]:
    f = invoice.fields.get(field_name)
    if f:
        inv[field_name] = date_to_dict(f)

# ── Currency fields ────────────────────────────────────────────
for field_name in [
    "SubTotal", "TotalDiscount", "TotalTax",
    "InvoiceTotal", "AmountDue", "PreviousUnpaidBalance",
]:
    f = invoice.fields.get(field_name)
    if f:
        inv[field_name] = currency_to_dict(f)

# ── PaymentDetails[] ───────────────────────────────────────────
payment_details = invoice.fields.get("PaymentDetails")
if payment_details:
    inv["PaymentDetails"] = []
    for detail in payment_details.value_array:
        obj = detail.value_object
        entry = {"location": get_bounding_info(detail)}
        for sub_field in ["IBAN", "SWIFT", "BankAccountNumber", "BPayBillerCode", "BPayReference"]:
            f = obj.get(sub_field)
            if f:
                entry[sub_field] = string_to_dict(f)
        inv["PaymentDetails"].append(entry)

# ── TaxDetails[] ──────────────────────────────────────────────
tax_details = invoice.fields.get("TaxDetails")
if tax_details:
    inv["TaxDetails"] = []
    for detail in tax_details.value_array:
        obj = detail.value_object
        entry = {"location": get_bounding_info(detail)}
        amount = obj.get("Amount")
        if amount:
            entry["Amount"] = currency_to_dict(amount)
        rate = obj.get("Rate")
        if rate:
            entry["Rate"] = string_to_dict(rate)
        inv["TaxDetails"].append(entry)

# ── PaidInFourInstallments[] ───────────────────────────────────
installments = invoice.fields.get("PaidInFourInstallements")
if installments:
    inv["PaidInFourInstallements"] = []
    for installment in installments.value_array:
        obj = installment.value_object
        entry = {"location": get_bounding_info(installment)}
        amount = obj.get("Amount")
        if amount:
            entry["Amount"] = currency_to_dict(amount)
        due_date = obj.get("DueDate")
        if due_date:
            entry["DueDate"] = date_to_dict(due_date)
        inv["PaidInFourInstallements"].append(entry)

# ── Items[] ───────────────────────────────────────────────────
items = invoice.fields.get("Items")
if items:
    inv["Items"] = []
    item_fields = {
        "Description": lambda f: f.value_string,
        "ProductCode": lambda f: f.value_string,
        "Unit":        lambda f: f.value_string,
        "TaxRate":     lambda f: f.value_string,
        "Quantity":    lambda f: f.value_number,
        "Date":        lambda f: str(f.value_date),
        "UnitPrice":   lambda f: f.value_currency.amount,
        "Tax":         lambda f: f.value_currency.amount,
        "Amount":      lambda f: f.value_currency.amount,
    }
    for item in items.value_array:
        obj = item.value_object
        item_dict = {"location": get_bounding_info(item)}
        for sub_field, getter in item_fields.items():
            f = obj.get(sub_field)
            if f:
                item_dict[sub_field] = {
                    "value": getter(f),
                    "confidence": f.confidence,
                    "location": get_bounding_info(f),
                }
        inv["Items"].append(item_dict)

# ── Wrap in list to keep downstream cells compatible ──────────
all_invoices = [inv]

print("Extraction complete")
# print(json.dumps(all_invoices, indent=3, default=str))

Extraction complete


In [9]:
with open("output/original.json", "w", encoding="utf-8") as f:
    json.dump(invoices.as_dict(), f, indent=3, default=str)

In [5]:
import os, re, json
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ── Fields your downstream pipeline requires ─────────────────────
REQUIRED_FIELDS = {
    "Currency":      "Currency",
    "VendorName":    "VendorName",
    "VendorAddress": "VendorAddress",
    "InvoiceId":     "InvoiceId",
    "InvoiceDate":   "InvoiceDate",
    "DueDate":       "DueDate",
    "PurchaseOrder": "PurchaseOrder",
    "PaymentTerm":   "PaymentTerm",
    "SubTotal":      "SubTotal",
    "TotalTax":      "TotalTax",
    "TotalDiscount": "TotalDiscount",
    "InvoiceTotal":  "InvoiceTotal",
    "IBAN":          None,
    "SWIFT":         None,
    "CustomerEmail": None,
}


# ─────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────

def wrap(value, confidence=None, location=None):
    return {"value": value, "confidence": confidence, "location": location or []}


def is_present(inv: dict, key: str) -> bool:
    v = inv.get(key)
    if v is None:
        return False
    if isinstance(v, dict):
        return v.get("value") is not None
    return True


def promote_payment_details(inv: dict) -> None:
    for detail in inv.get("PaymentDetails", []):
        for key in ("IBAN", "SWIFT"):
            if not is_present(inv, key) and key in detail:
                inv[key] = detail[key]


# ─────────────────────────────────────────────────────────────────
# STEP 1 — check what is already in inv after your extractor ran
# ─────────────────────────────────────────────────────────────────

def find_missing(inv: dict) -> set[str]:
    promote_payment_details(inv)
    missing = set()
    for canonical in REQUIRED_FIELDS:
        if not is_present(inv, canonical):
            missing.add(canonical)
    return missing


# ─────────────────────────────────────────────────────────────────
# STEP 2 — ask OpenAI to find only the missing fields
# ─────────────────────────────────────────────────────────────────

def ask_openai(missing_fields: set[str], adi_result: dict) -> dict:
    """
    Returns:
    {
      "FieldName": {
        "value":      <extracted value or null>,
        "confidence": <from ADI if found, else null>,
        "location":   [{"page": N, "bbox": [[x,y],...]}]
      },
      ...
    }
    """
    target = {k: None for k in missing_fields}

    prompt = f"""You are an invoice data extraction assistant.

You will receive the raw JSON from Azure Document Intelligence (ADI) for an invoice.
Find each field in TARGET FIELDS using ONLY data already present in the ADI JSON.

Rules:
1. Search everywhere: top-level fields, nested objects, Items[], PaymentDetails[], content strings.
2. Use the FIRST occurrence of each field only.
3. Return the ADI field's own `confidence` value if available, else null.
4. Return coordinates from the ADI field's own `boundingRegions` converted to:
   "location": [{{"page": <pageNumber>, "bbox": [[x0,y0],[x1,y1],[x2,y2],[x3,y3]]}}]
   Polygon in ADI is a flat list [x0,y0,x1,y1,...] — convert to pairs.
   If no boundingRegions exist, return "location": [].
5. DO NOT invent values or coordinates. If a field is absent, return value: null.
6. Field-specific hints:
   - Currency     → find any valueCurrency.currencyCode (e.g. "USD")
   - CustomerEmail → look for BillingEmail or any email associated with the customer
   - IBAN / SWIFT → look inside PaymentDetails array
7. Return ONLY valid JSON. No explanation, no markdown, no code fences.

### TARGET FIELDS:
{json.dumps(target, indent=2)}

### ADI RESULT:
{json.dumps(adi_result, indent=2, default=str)}
"""

    resp = client.chat.completions.create(
        model="gpt-4.1",          # swap to "gpt-4-turbo" or "gpt-3.5-turbo" as needed
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )

    raw   = resp.choices[0].message.content
    clean = re.sub(r"```(?:json)?|```", "", raw).strip()
    return json.loads(clean)


# ─────────────────────────────────────────────────────────────────
# STEP 3 — merge OpenAI result back into inv
# ─────────────────────────────────────────────────────────────────

def merge_openai_result(inv: dict, openai_result: dict) -> None:
    for field_name, data in openai_result.items():
        if not isinstance(data, dict):
            inv[field_name] = wrap(data)
            continue

        value      = data.get("value")
        confidence = data.get("confidence")
        raw_loc    = data.get("location", [])

        location = [
            {
                "page": loc.get("page"),
                "bbox": [tuple(pt) if isinstance(pt, list) else pt
                         for pt in loc.get("bbox", [])],
            }
            for loc in raw_loc
        ]

        entry = wrap(value, confidence, location)
        print(f"  [OpenAI] {field_name} = {value}  (conf={confidence})")

        if field_name in ("IBAN", "SWIFT") and value:
            if not inv.get("PaymentDetails"):
                inv["PaymentDetails"] = [{}]
            inv["PaymentDetails"][0].setdefault(field_name, entry)

        inv[field_name] = entry


# ─────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────

def enrich_invoice(inv: dict, adi_result: dict) -> dict:
    """
    inv        — output of your existing ADI extractor (all_invoices[0])
    adi_result — raw ADI poller dict (invoices.documents[0] serialised,
                 or the raw dict from the SDK before your field parsing)
    """
    print("Checking required fields …")
    missing = find_missing(inv)
    print(f"  Present : {sorted(REQUIRED_FIELDS.keys() - missing)}")
    print(f"  Missing : {sorted(missing)}")

    if missing:
        print("Calling OpenAI for missing fields …")
        openai_result = ask_openai(missing, adi_result)
        merge_openai_result(inv, openai_result)
    else:
        print("All required fields present — skipping OpenAI call.")

    return inv


# ── Usage ──────────────────────────────────────────────────────────
# inv = all_invoices[0]
# inv = enrich_invoice(inv, adi_result)

inv = enrich_invoice(inv, invoices.as_dict())

print("\nFinal invoice:")
print(json.dumps(inv, indent=2, default=str))

Checking required fields …
  Present : ['Currency', 'DueDate', 'InvoiceDate', 'InvoiceId', 'InvoiceTotal', 'PaymentTerm', 'PurchaseOrder', 'SubTotal', 'TotalTax', 'VendorAddress', 'VendorName']
  Missing : ['CustomerEmail', 'IBAN', 'SWIFT', 'TotalDiscount']
Calling OpenAI for missing fields …


RateLimitError: Error code: 429 - {'error': {'message': 'Request too large for gpt-4.1 in organization org-8zoxeY4EXNX5WQPhliF0Ynjq on tokens per min (TPM): Limit 30000, Requested 62869. The input or output tokens must be reduced in order to run successfully. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

In [ ]:
import os, re, json, requests

MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")
MISTRAL_API_URL = "https://api.mistral.ai/v1/chat/completions"

# ── Fields your downstream pipeline requires ─────────────────────
# Map: canonical name → where your extractor already puts it (or None)
REQUIRED_FIELDS = {
    "Currency":      "Currency",       # top-level scalar
    "VendorName":    "VendorName",
    "VendorAddress": "VendorAddress",
    "InvoiceId":     "InvoiceId",
    "InvoiceDate":   "InvoiceDate",
    "DueDate":       "DueDate",
    "PurchaseOrder": "PurchaseOrder",
    "PaymentTerm":   "PaymentTerm",
    "SubTotal":      "SubTotal",
    "TotalTax":      "TotalTax",
    "TotalDiscount": "TotalDiscount",
    "InvoiceTotal":  "InvoiceTotal",
    "IBAN":          None,             # nested in PaymentDetails[]
    "SWIFT":         None,             # nested in PaymentDetails[]
    "CustomerEmail": None,             # ADI field BillingEmail — not extracted yet
}


# ─────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────

def wrap(value, confidence=None, location=None):
    return {"value": value, "confidence": confidence, "location": location or []}


def is_present(inv: dict, key: str) -> bool:
    v = inv.get(key)
    if v is None:
        return False
    if isinstance(v, dict):
        return v.get("value") is not None
    return True  # plain scalar e.g. Currency = "USD"


def promote_payment_details(inv: dict) -> None:
    """
    Hoist the first IBAN / SWIFT found in PaymentDetails[]
    up to the top level so downstream checks are uniform.
    """
    for detail in inv.get("PaymentDetails", []):
        for key in ("IBAN", "SWIFT"):
            if not is_present(inv, key) and key in detail:
                inv[key] = detail[key]


# ─────────────────────────────────────────────────────────────────
# STEP 1 — check what is already in inv after your extractor ran
# ─────────────────────────────────────────────────────────────────

def find_missing(inv: dict) -> set[str]:
    promote_payment_details(inv)

    missing = set()
    for canonical in REQUIRED_FIELDS:
        if not is_present(inv, canonical):
            missing.add(canonical)

    return missing


# ─────────────────────────────────────────────────────────────────
# STEP 2 — ask Mistral to find only the missing fields
#           using the full ADI poller result as its source
# ─────────────────────────────────────────────────────────────────

def ask_mistral(missing_fields: set[str], adi_result: dict) -> dict:
    """
    Mistral receives the raw ADI JSON and a target schema.
    It must find each missing field inside the ADI JSON —
    reading existing boundingRegions for coordinates, NOT inventing them.

    Returns:
    {
      "FieldName": {
        "value":      <extracted value or null>,
        "confidence": <from ADI if found, else null>,
        "location":   [{"page": N, "bbox": [[x,y],...]}]  # first occurrence only
      },
      ...
    }
    """
    target = {k: None for k in missing_fields}

    prompt = f"""You are an invoice data extraction assistant.

You will receive the raw JSON from Azure Document Intelligence (ADI) for an invoice.
Find each field in TARGET FIELDS using ONLY data already present in the ADI JSON.

Rules:
1. Search everywhere: top-level fields, nested objects, Items[], PaymentDetails[], content strings.
2. Use the FIRST occurrence of each field only.
3. Return the ADI field's own `confidence` value if available, else null.
4. Return coordinates from the ADI field's own `boundingRegions` converted to:
   "location": [{{"page": <pageNumber>, "bbox": [[x0,y0],[x1,y1],[x2,y2],[x3,y3]]}}]
   Polygon in ADI is a flat list [x0,y0,x1,y1,...] — convert to pairs.
   If no boundingRegions exist, return "location": [].
5. DO NOT invent values or coordinates. If a field is absent, return value: null.
6. Field-specific hints:
   - Currency    → find any valueCurrency.currencyCode (e.g. "USD")
   - CustomerEmail → look for BillingEmail or any email associated with the customer
   - IBAN / SWIFT → look inside PaymentDetails array
7. Return ONLY valid JSON. No explanation, no markdown, no code fences.

### TARGET FIELDS:
{json.dumps(target, indent=2)}

### ADI RESULT:
{json.dumps(adi_result, indent=2, default=str)}
"""

    resp = requests.post(
        MISTRAL_API_URL,
        headers={
            "Authorization": f"Bearer {MISTRAL_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model":       "mistral-large-latest",
            "temperature": 0,
            "messages":    [{"role": "user", "content": prompt}],
        },
        timeout=120,
    )
    resp.raise_for_status()
    raw   = resp.json()["choices"][0]["message"]["content"]
    clean = re.sub(r"```(?:json)?|```", "", raw).strip()
    return json.loads(clean)


# ─────────────────────────────────────────────────────────────────
# STEP 3 — merge Mistral result back into inv
# ─────────────────────────────────────────────────────────────────

def merge_mistral_result(inv: dict, mistral_result: dict) -> None:
    for field_name, data in mistral_result.items():
        if not isinstance(data, dict):
            inv[field_name] = wrap(data)
            continue

        value      = data.get("value")
        confidence = data.get("confidence")
        raw_loc    = data.get("location", [])

        location = [
            {
                "page": loc.get("page"),
                "bbox": [tuple(pt) if isinstance(pt, list) else pt
                         for pt in loc.get("bbox", [])],
            }
            for loc in raw_loc
        ]

        entry = wrap(value, confidence, location)
        print(f"  [Mistral] {field_name} = {value}  (conf={confidence})")

        # IBAN / SWIFT also mirror into PaymentDetails for downstream compat
        if field_name in ("IBAN", "SWIFT") and value:
            if not inv.get("PaymentDetails"):
                inv["PaymentDetails"] = [{}]
            inv["PaymentDetails"][0].setdefault(field_name, entry)

        inv[field_name] = entry


# ─────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────

def enrich_invoice(inv: dict, adi_result: dict) -> dict:
    """
    inv        — output of your existing ADI extractor (all_invoices[0])
    adi_result — raw ADI poller dict (invoices.documents[0] serialised,
                 or the raw dict from the SDK before your field parsing)
    """
    print("Checking required fields …")
    missing = find_missing(inv)
    print(f"  Present : {sorted(REQUIRED_FIELDS.keys() - missing)}")
    print(f"  Missing : {sorted(missing)}")

    if missing:
        print("Calling Mistral for missing fields …")
        mistral_result = ask_mistral(missing, adi_result)
        merge_mistral_result(inv, mistral_result)
    else:
        print("All required fields present — skipping Mistral call.")

    return inv


# ── Usage ──────────────────────────────────────────────────────────
# Run your existing extractor first, then:
#
#   inv = all_invoices[0]
#   inv = enrich_invoice(inv, adi_result)
#
# `adi_result` is the raw ADI dict — either:
#   a) the dict you already have (the one you pasted at the top)
#   b) or serialise the SDK object:  adi_result = invoice.to_dict()

inv = enrich_invoice(inv, invoice)

print("\nFinal invoice:")
print(json.dumps(inv, indent=2, default=str))

In [ ]:
import fitz  # PyMuPDF
import requests
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from PIL import Image
from io import BytesIO
import numpy as np
import os

# ── Config ─────────────────────────────────────────────────────────
IMAGE_SOURCE = formUrl  # URL or local path — works for both PDF and JPG/PNG
RENDER_DPI   = 150

CATEGORY_COLORS = {
    "string":   "#2196F3",
    "address":  "#4CAF50",
    "date":     "#FF9800",
    "currency": "#9C27B0",
    "array":    "#F44336",
}

FIELD_CATEGORIES = {
    **{k: "string" for k in [
        "VendorName", "VendorAddressRecipient", "CustomerName", "CustomerId",
        "CustomerAddressRecipient", "BillingAddressRecipient", "ShippingAddressRecipient",
        "RemittanceAddressRecipient", "ServiceAddressRecipient", "PurchaseOrder",
        "InvoiceId", "VendorTaxId", "CustomerTaxId", "PaymentTerm",
    ]},
    **{k: "address" for k in [
        "VendorAddress", "CustomerAddress", "BillingAddress", "ShippingAddress",
        "RemittanceAddress", "ServiceAddress",
    ]},
    **{k: "date" for k in [
        "InvoiceDate", "DueDate", "ServiceStartDate", "ServiceEndDate",
    ]},
    **{k: "currency" for k in [
        "SubTotal", "TotalDiscount", "TotalTax", "InvoiceTotal",
        "AmountDue", "PreviousUnpaidBalance",
    ]},
    **{k: "array" for k in [
        "PaymentDetails", "TaxDetails", "Items",
    ]},
}


# ── Source loader ───────────────────────────────────────────────────
def load_bytes(source):
    if source.startswith("http://") or source.startswith("https://"):
        resp = requests.get(source)
        resp.raise_for_status()
        return resp.content
    with open(source, "rb") as f:
        return f.read()

def is_pdf(source, raw_bytes):
    ext = os.path.splitext(source.split("?")[0])[1].lower()
    if ext == ".pdf":
        return True
    if ext in (".jpg", ".jpeg", ".png", ".tiff", ".bmp", ".webp"):
        return False
    return raw_bytes[:4] == b"%PDF"

def get_page_images(source):
    raw = load_bytes(source)
    pages = {}
    if is_pdf(source, raw):
        doc = fitz.open(stream=raw, filetype="pdf")
        scale = RENDER_DPI / 72
        mat = fitz.Matrix(scale, scale)
        for i, page in enumerate(doc):
            pix = page.get_pixmap(matrix=mat)
            img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
            if pix.n == 4:
                img = img[:, :, :3]
            pages[i + 1] = (img, RENDER_DPI)
        doc.close()
    else:
        img = np.array(Image.open(BytesIO(raw)).convert("RGB"))
        pages[1] = (img, 1)
    return pages


# ── Region collector ───────────────────────────────────────────────
def collect_all_regions_from_dict(invoice_dict):
    page_regions = {}

    def add_location(location_list, label, color):
        if not location_list:
            return
        for loc in location_list:
            page = loc.get("page")
            bbox = loc.get("bbox")
            if page is None or not bbox:
                continue
            page_regions.setdefault(page, []).append((bbox, label, color))

    def process_field(field_value, label, color):
        if not isinstance(field_value, dict):
            return
        add_location(field_value.get("location", []), label, color)

    # ── Scalar fields ─────────────────────────────────────────────
    for field_name, category in FIELD_CATEGORIES.items():
        if category == "array":
            continue
        field = invoice_dict.get(field_name)
        if field:
            process_field(field, field_name, CATEGORY_COLORS[category])

    # ── Array fields ──────────────────────────────────────────────
    array_color = CATEGORY_COLORS["array"]
    for arr_field_name in ["PaymentDetails", "TaxDetails", "Items"]:
        arr = invoice_dict.get(arr_field_name)
        if not arr or not isinstance(arr, list):
            continue

        prefix = arr_field_name.rstrip("s")

        for i, item in enumerate(arr):
            if not isinstance(item, dict):
                continue
            item_label = "{}[{}]".format(prefix, i)

            for sub_name, sub_field in item.items():
                if sub_name in ("location", "bbox"):
                    continue
                sub_label = "{}.{}".format(item_label, sub_name)
                process_field(sub_field, sub_label, array_color)

    return page_regions


# ── Plotter ─────────────────────────────────────────────────────────
def plot_page(page_num, img, scale, regions):
    fig, ax = plt.subplots(figsize=(14, 18))
    ax.imshow(img)
    ax.set_title("Page {} — Detected Invoice Fields".format(page_num), fontsize=14)
    ax.axis("off")

    for coords, label, color in regions:
        scaled = [(x * scale, y * scale) for x, y in coords]
        xs = [p[0] for p in scaled] + [scaled[0][0]]
        ys = [p[1] for p in scaled] + [scaled[0][1]]
        ax.plot(xs, ys, color=color, linewidth=1.5)
        ax.text(scaled[0][0], scaled[0][1] - 4, label,
                fontsize=5.5, color=color, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.1", fc="white", ec=color, alpha=0.7))

    legend_elements = [
        Line2D([0], [0], color=c, linewidth=2, label=cat.capitalize())
        for cat, c in CATEGORY_COLORS.items()
    ]
    ax.legend(handles=legend_elements, loc="upper right", fontsize=8)
    plt.tight_layout()

    out = "output/invoice_page_{}_annotated.png".format(page_num)
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: {}".format(out))


# ── Run ─────────────────────────────────────────────────────────────
page_images  = get_page_images(IMAGE_SOURCE)
page_regions = collect_all_regions_from_dict(inv)

for page_num in sorted(page_images):
    img, scale = page_images[page_num]
    regions = page_regions.get(page_num, [])
    plot_page(page_num, img, scale, regions)

In [ ]:
json.dump(inv, open(output_dir / "final_invoice.json", "w"), indent=2, default=str)

In [ ]:
type(all_invoices[0])